In [ ]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

In [ ]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [ ]:
# Keep users and books with enough ratings for meaningful comparisons.
ratings_by_user = df_ratings['user'].value_counts()
ratings_by_book = df_ratings['isbn'].value_counts()
active_users = ratings_by_user[ratings_by_user >= 200].index
popular_books = ratings_by_book[ratings_by_book >= 100].index

filtered_ratings = df_ratings[
    df_ratings['user'].isin(active_users) & df_ratings['isbn'].isin(popular_books)
].copy()

# Rows are books and columns are users; CSR keeps the mostly-empty matrix compact.
book_user_pivot = filtered_ratings.pivot_table(
    index='isbn', columns='user', values='rating'
).fillna(0)
book_user_matrix = csr_matrix(book_user_pivot.values)

# Map ISBNs to display titles, preserving matrix row order.
isbn_to_title = df_books.drop_duplicates('isbn').set_index('isbn')['title']
book_titles = isbn_to_title.reindex(book_user_pivot.index)


In [ ]:
# function to return recommended books - this will be tested
def get_recommends(book=""):
    matches = np.flatnonzero(book_titles.to_numpy() == book)
    if len(matches) == 0:
        raise ValueError(f'Book title not found in the filtered dataset: {book}')

    model = NearestNeighbors(metric='cosine', algorithm='brute')
    model.fit(book_user_matrix)
    distances, indices = model.kneighbors(
        book_user_matrix[matches[0]], n_neighbors=6
    )

    recommendations = []
    # Reverse nearest-neighbor order to match the project test's expected list.
    for distance, index in zip(distances[0][1:], indices[0][1:]):
        title = book_titles.iloc[index]
        recommendations.append([title, float(distance)])
    recommendations.reverse()
    return [book, recommendations]


In [ ]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2): 
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()